# Supply Chain Deliveries Analysis & Revenue Prediction

Exploratory analysis and a reproducible baseline revenue model.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("../data/supply_chain_deliveries.csv")
df = pd.read_csv(DATA, parse_dates=["WorkDate"])
df.head()


## Data quality and summary


In [ ]:
print(df.shape)
print(df.isna().sum())
df.describe(include="all")


## Revenue over time


In [ ]:
monthly = df.set_index("WorkDate").resample("MS")["TotalRevenue"].sum()
ax = monthly.plot(figsize=(12,5), title="Monthly Revenue")
ax.set_ylabel("Revenue ($)")
plt.show()


## Customer performance


In [ ]:
df.groupby("Customer")["TotalRevenue"].sum().sort_values().plot.barh(figsize=(9,6), title="Revenue by Customer")
plt.xlabel("Revenue ($)")
plt.show()


## Location and business type performance


In [ ]:
display(df.groupby("Location")["TotalRevenue"].sum().sort_values(ascending=False).to_frame("Revenue"))
display(df.groupby("BusinessType")[["OrderCount","NumberOfPieces","TotalRevenue"]].sum().sort_values("TotalRevenue", ascending=False))


## Baseline revenue prediction

Use a chronological split and a random forest with one-hot encoded categorical features.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

m = df.sort_values("WorkDate").copy()
m["Year"] = m.WorkDate.dt.year
m["Month"] = m.WorkDate.dt.month
m["DayOfWeek"] = m.WorkDate.dt.dayofweek
features = ["Customer","Location","BusinessType","OrderCount","NumberOfPieces","Year","Month","DayOfWeek"]
cat = ["Customer","Location","BusinessType"]
num = [x for x in features if x not in cat]
split_date = m.WorkDate.quantile(.80)
train, test = m[m.WorkDate <= split_date], m[m.WorkDate > split_date]
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat),("num", "passthrough", num)])
pipe = Pipeline([("prep", prep),("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1, max_depth=18))])
pipe.fit(train[features], train.TotalRevenue)
pred = pipe.predict(test[features])
print("MAE:", mean_absolute_error(test.TotalRevenue,pred))
print("RMSE:", mean_squared_error(test.TotalRevenue,pred)**0.5)
print("R2:", r2_score(test.TotalRevenue,pred))


## Next steps

Add lag features, rolling statistics, time-series backtesting, explainability, and an interactive dashboard.
